# IBM AML Compliance Project — Google Colab

This notebook is configured for Kaggle's newer **`KGAT_...` API token** authentication.

## Security

- The token is requested through `getpass()`, so it is not displayed in notebook output.
- The token is not hard-coded into this notebook.
- The token exists only for the current Colab runtime.
- Never save the token in Google Drive, GitHub, notebook text, screenshots, or public outputs.
- If a token has been shared publicly, revoke it in Kaggle and generate a new token before running this notebook.

## Notebook workflow

1. Install Kaggle and project packages
2. Enter the `KGAT_...` token securely
3. Verify Kaggle authentication
4. Download the IBM AML dataset
5. Extract and locate `HI-Small_Trans.csv`
6. Load and validate a development sample
7. Perform exploratory analysis
8. Engineer baseline behavioural features
9. Create chronological train, validation and test sets
10. Train and evaluate a Logistic Regression model
11. Save outputs to Google Drive


## 1. Install packages

In [1]:
!pip install -q --upgrade kaggle
!pip install -q pyarrow joblib scikit-learn pandas numpy matplotlib

## 2. Enter your Kaggle `KGAT_...` token securely

Paste a newly generated token when prompted. The input will be hidden.

Token = KGAT_8d582a47bb55cd03e5b7837ec3432fd5


In [2]:
import os
from getpass import getpass

kaggle_token = getpass("Paste your Kaggle KGAT token: ").strip()

if not kaggle_token:
    raise ValueError("No Kaggle token was provided.")

if not kaggle_token.startswith("KGAT_"):
    print(
        "Warning: This token does not start with KGAT_. "
        "Confirm that you copied the current Kaggle API token."
    )

os.environ["KAGGLE_API_TOKEN"] = kaggle_token

print("Kaggle token configured for this Colab runtime.")


Paste your Kaggle KGAT token:  ········


Kaggle token configured for this Colab runtime.


## 3. Test Kaggle authentication

In [3]:
!kaggle datasets list -s "IBM transactions anti money laundering"

ref                                                         title                                                   size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------  ------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
ealtman2019/ibm-transactions-for-anti-money-laundering-aml  IBM Transactions for Anti Money Laundering (AML)  8176169418  2025-07-08 14:59:13.280000          35799        256  0.9411765        
anshankul/ibm-amlsim-example-dataset                        IBM AMLSim Example Dataset                          11924068  2021-07-15 01:30:25.003000           3808         39  0.8235294        
ealtman2019/credit-card-transactions                        Credit Card Transactions                           276210511  2021-10-14 17:42:24.357000          24302        160  0.85294116       
navpr06/consent-managed-transa

## 4. Create project directories

In [4]:
from pathlib import Path

BASE_DIR = Path("/content/ibm_aml")
DOWNLOAD_DIR = BASE_DIR / "download"
EXTRACT_DIR = BASE_DIR / "data"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Download directory:", DOWNLOAD_DIR)
print("Extraction directory:", EXTRACT_DIR)


Download directory: \content\ibm_aml\download
Extraction directory: \content\ibm_aml\data


## 5. Download the IBM AML dataset from Kaggle

In [5]:
!kaggle datasets download     -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml     -p "$DOWNLOAD_DIR" 


 10%|#         | 781M/7.61G [00:00<?, ?B/s]
 10%|#         | 782M/7.61G [00:01<3:17:05, 622kB/s]
 10%|#         | 783M/7.61G [00:01<1:38:21, 1.25MB/s]
 10%|#         | 784M/7.61G [00:02<1:10:09, 1.75MB/s]
 10%|#         | 785M/7.61G [00:02<53:09, 2.31MB/s]  
 10%|#         | 786M/7.61G [00:02<45:17, 2.71MB/s]
 10%|#         | 787M/7.61G [00:03<40:09, 3.05MB/s]
 10%|#         | 788M/7.61G [00:03<38:14, 3.20MB/s]
 10%|#         | 789M/7.61G [00:03<36:00, 3.40MB/s]
 10%|#         | 790M/7.61G [00:03<35:17, 3.47MB/s]
 10%|#         | 791M/7.61G [00:04<34:35, 3.54MB/s]
 10%|#         | 792M/7.61G [00:04<33:46, 3.63MB/s]
 10%|#         | 793M/7.61G [00:04<33:37, 3.64MB/s]
 10%|#         | 794M/7.61G [00:04<32:54, 3.72MB/s]
 10%|#         | 795M/7.61G [00:05<31:51, 3.84MB/s]
 10%|#         | 796M/7.61G [00:05<27:13, 4.49MB/s]
 10%|#         | 797M/7.61G [00:05<25:26, 4.81MB/s]
 10%|#         | 798M/7.61G [00:05<22:53, 5.34MB/s]
 10%|#         | 799M/7.61G [00:05<22:18, 5.48MB/s]
 10%|#      

Dataset URL: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
License(s): Community Data License Agreement - Sharing - Version 1.0
Resuming from 818937856 bytes (7357231562 bytes left)...



 78%|#######8  | 5.94G/7.61G [18:42<06:16, 4.78MB/s]
 78%|#######8  | 5.94G/7.61G [18:42<05:43, 5.23MB/s]
 78%|#######8  | 5.94G/7.61G [18:42<05:09, 5.79MB/s]
 78%|#######8  | 5.94G/7.61G [18:42<04:44, 6.29MB/s]
 78%|#######8  | 5.95G/7.61G [18:43<04:31, 6.61MB/s]
 78%|#######8  | 5.95G/7.61G [18:43<05:03, 5.90MB/s]
 78%|#######8  | 5.95G/7.61G [18:43<06:01, 4.96MB/s]
 78%|#######8  | 5.95G/7.61G [18:43<05:29, 5.44MB/s]
 78%|#######8  | 5.95G/7.61G [18:43<05:18, 5.61MB/s]
 78%|#######8  | 5.95G/7.61G [18:44<06:20, 4.70MB/s]
 78%|#######8  | 5.95G/7.61G [18:44<06:42, 4.43MB/s]
 78%|#######8  | 5.95G/7.61G [18:44<05:52, 5.06MB/s]
 78%|#######8  | 5.95G/7.61G [18:44<05:32, 5.37MB/s]
 78%|#######8  | 5.95G/7.61G [18:44<05:02, 5.90MB/s]
 78%|#######8  | 5.96G/7.61G [18:45<04:45, 6.25MB/s]
 78%|#######8  | 5.96G/7.61G [18:45<05:22, 5.52MB/s]
 78%|#######8  | 5.96G/7.61G [18:45<05:01, 5.91MB/s]
 78%|#######8  | 5.96G/7.61G [18:45<06:10, 4.81MB/s]
 78%|#######8  | 5.96G/7.61G [18:46<06:42, 4.4

In [6]:
!ls -lh "$DOWNLOAD_DIR" 

'ls' is not recognized as an internal or external command,
operable program or batch file.


## 6. Extract the downloaded archive and any nested archives

In [7]:
import glob
import os
import zipfile

downloaded_zip_files = glob.glob(str(DOWNLOAD_DIR / "*.zip"))

if not downloaded_zip_files:
    raise FileNotFoundError(
        "No ZIP file was downloaded. Re-run the authentication test and download cells."
    )

for zip_path in downloaded_zip_files:
    print("Extracting:", os.path.basename(zip_path))
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(EXTRACT_DIR)

nested_zip_files = glob.glob(
    str(EXTRACT_DIR / "**" / "*.zip"),
    recursive=True
)

for zip_path in nested_zip_files:
    target_dir = Path(zip_path).with_suffix("")
    target_dir.mkdir(parents=True, exist_ok=True)

    print("Extracting nested archive:", zip_path)

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(target_dir)

print("Extraction completed.")


Extracting: ibm-transactions-for-anti-money-laundering-aml.zip


OSError: [Errno 28] No space left on device

## 7. Inventory the available CSV files

In [ ]:
import pandas as pd

csv_files = glob.glob(
    str(EXTRACT_DIR / "**" / "*.csv"),
    recursive=True
)

if not csv_files:
    raise FileNotFoundError(
        "No CSV files were found after extraction."
    )

file_inventory = pd.DataFrame(
    [
        {
            "filename": os.path.basename(path),
            "path": path,
            "size_mb": round(
                os.path.getsize(path) / (1024 ** 2),
                2
            ),
        }
        for path in csv_files
    ]
).sort_values("size_mb")

file_inventory.reset_index(drop=True)


## 8. Locate the HI-Small transaction file

The notebook searches by filename rather than assuming a fixed directory structure.


In [ ]:
hi_small_candidates = [
    path
    for path in csv_files
    if "hi-small" in os.path.basename(path).lower()
    and "trans" in os.path.basename(path).lower()
]

if not hi_small_candidates:
    raise FileNotFoundError(
        "HI-Small transaction file was not found. "
        "Review the CSV inventory shown above."
    )

DATA_PATH = sorted(
    hi_small_candidates,
    key=os.path.getsize
)[0]

print("Selected transaction file:")
print(DATA_PATH)
print(
    "File size:",
    round(os.path.getsize(DATA_PATH) / (1024 ** 2), 2),
    "MB"
)


## 9. Preview the schema

In [ ]:
preview = pd.read_csv(
    DATA_PATH,
    nrows=5,
    low_memory=False
)

print("Available columns:")
print(preview.columns.tolist())

preview


## 10. Load a development sample

Start with 200,000 rows in a standard Colab runtime. Increase this after the notebook runs successfully.


In [ ]:
N_ROWS = 200_000

df = pd.read_csv(
    DATA_PATH,
    nrows=N_ROWS,
    low_memory=False
)

print("Dataset shape:", df.shape)
print(
    "Approximate memory:",
    round(
        df.memory_usage(deep=True).sum() / (1024 ** 2),
        2
    ),
    "MB"
)

df.head()


## 11. Rename and validate columns

In [ ]:
column_mapping = {
    "Timestamp": "event_timestamp",
    "From Bank": "sender_bank_id",
    "Account": "sender_account_id",
    "To Bank": "receiver_bank_id",
    "Account.1": "receiver_account_id",
    "Amount Received": "amount_received",
    "Receiving Currency": "receiving_currency",
    "Amount Paid": "amount_paid",
    "Payment Currency": "payment_currency",
    "Payment Format": "payment_format",
    "Is Laundering": "is_laundering",
}

df = df.rename(columns=column_mapping)

required_columns = {
    "event_timestamp",
    "sender_bank_id",
    "sender_account_id",
    "receiver_bank_id",
    "receiver_account_id",
    "amount_received",
    "receiving_currency",
    "amount_paid",
    "payment_currency",
    "payment_format",
    "is_laundering",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Required columns were not found: {sorted(missing_columns)}"
    )

print("Schema validation passed.")


## 12. Convert data types

In [ ]:
df["event_timestamp"] = pd.to_datetime(
    df["event_timestamp"],
    errors="coerce"
)

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"],
    errors="coerce"
)

df["amount_received"] = pd.to_numeric(
    df["amount_received"],
    errors="coerce"
)

df["is_laundering"] = pd.to_numeric(
    df["is_laundering"],
    errors="coerce"
).fillna(0).astype("int8")

df.info()


## 13. Data-quality report

In [ ]:
quality_report = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "duplicate_rows",
            "invalid_timestamps",
            "missing_sender_accounts",
            "missing_receiver_accounts",
            "missing_amount_paid",
            "non_positive_amount_paid",
            "laundering_rows",
            "laundering_rate",
        ],
        "value": [
            len(df),
            df.shape[1],
            int(df.duplicated().sum()),
            int(df["event_timestamp"].isna().sum()),
            int(df["sender_account_id"].isna().sum()),
            int(df["receiver_account_id"].isna().sum()),
            int(df["amount_paid"].isna().sum()),
            int((df["amount_paid"] <= 0).sum()),
            int(df["is_laundering"].sum()),
            float(df["is_laundering"].mean()),
        ],
    }
)

quality_report


In [ ]:
missing_report = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)

missing_report["missing_percentage"] = (
    missing_report["missing_count"]
    / len(df)
    * 100
)

missing_report.head(20)


## 14. Analyse class imbalance

In [ ]:
label_summary = (
    df["is_laundering"]
    .value_counts(dropna=False)
    .sort_index()
    .rename(
        index={
            0: "Legitimate",
            1: "Laundering"
        }
    )
    .to_frame("count")
)

label_summary["percentage"] = (
    label_summary["count"]
    / label_summary["count"].sum()
    * 100
)

label_summary


In [ ]:
import matplotlib.pyplot as plt

label_summary["count"].plot(kind="bar")

plt.title("IBM AML Class Distribution")
plt.xlabel("Class")
plt.ylabel("Transaction Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 15. Initial exploratory summaries

In [ ]:
payment_format_summary = (
    df.groupby("payment_format")
      .agg(
          transaction_count=("is_laundering", "size"),
          laundering_count=("is_laundering", "sum"),
          laundering_rate=("is_laundering", "mean"),
      )
      .sort_values(
          "transaction_count",
          ascending=False
      )
)

payment_format_summary


In [ ]:
currency_summary = (
    df.groupby("payment_currency")
      .agg(
          transaction_count=("is_laundering", "size"),
          laundering_count=("is_laundering", "sum"),
          laundering_rate=("is_laundering", "mean"),
      )
      .sort_values(
          "transaction_count",
          ascending=False
      )
)

currency_summary.head(20)


## 16. Create transaction and account node identifiers

In [ ]:
df = df.reset_index(drop=True)

df["transaction_id"] = (
    "TXN-"
    + df.index.astype(str).str.zfill(10)
)

df["sender_node_id"] = (
    df["sender_bank_id"].astype(str)
    + "_"
    + df["sender_account_id"].astype(str)
)

df["receiver_node_id"] = (
    df["receiver_bank_id"].astype(str)
    + "_"
    + df["receiver_account_id"].astype(str)
)

df[
    [
        "transaction_id",
        "sender_node_id",
        "receiver_node_id",
    ]
].head()


## 17. Engineer basic transaction features

In [ ]:
import numpy as np

df["log_amount_paid"] = np.log1p(
    df["amount_paid"].clip(lower=0)
)

df["amount_difference"] = (
    df["amount_received"]
    - df["amount_paid"]
).abs()

df["log_amount_difference"] = np.log1p(
    df["amount_difference"]
)

df["currency_mismatch"] = (
    df["payment_currency"]
    != df["receiving_currency"]
).astype("int8")

df["is_cross_bank"] = (
    df["sender_bank_id"]
    != df["receiver_bank_id"]
).astype("int8")

df["transaction_hour"] = (
    df["event_timestamp"].dt.hour
)

df["day_of_week"] = (
    df["event_timestamp"].dt.dayofweek
)

df["is_weekend"] = (
    df["day_of_week"].isin([5, 6])
).astype("int8")


## 18. Engineer sender-history and new-counterparty features

In [ ]:
df = df.sort_values(
    [
        "sender_node_id",
        "event_timestamp"
    ]
).reset_index(drop=True)

df["sender_previous_transaction_count"] = (
    df.groupby("sender_node_id")
      .cumcount()
)

sender_group = df.groupby(
    "sender_node_id",
    group_keys=False
)

df["sender_historical_average_amount"] = (
    sender_group["amount_paid"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(
        level=0,
        drop=True
    )
)

df["amount_to_sender_average_ratio"] = (
    df["amount_paid"]
    / df[
        "sender_historical_average_amount"
    ].replace(0, np.nan)
)

df["amount_to_sender_average_ratio"] = (
    df["amount_to_sender_average_ratio"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(1.0)
)


In [ ]:
df["sender_receiver_pair"] = (
    df["sender_node_id"]
    + "→"
    + df["receiver_node_id"]
)

df["previous_pair_transaction_count"] = (
    df.groupby(
        "sender_receiver_pair"
    ).cumcount()
)

df["new_counterparty"] = (
    df["previous_pair_transaction_count"] == 0
).astype("int8")


## 19. Review engineered features

In [ ]:
engineered_columns = [
    "transaction_id",
    "event_timestamp",
    "amount_paid",
    "log_amount_paid",
    "amount_difference",
    "currency_mismatch",
    "is_cross_bank",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
    "sender_previous_transaction_count",
    "sender_historical_average_amount",
    "amount_to_sender_average_ratio",
    "new_counterparty",
    "is_laundering",
]

df[engineered_columns].head(10)


## 20. Create chronological train, validation and test sets

In [ ]:
df = df.sort_values(
    "event_timestamp"
).reset_index(drop=True)

n_rows = len(df)

train_end = int(n_rows * 0.60)
validation_end = int(n_rows * 0.80)

train_df = df.iloc[
    :train_end
].copy()

validation_df = df.iloc[
    train_end:validation_end
].copy()

test_df = df.iloc[
    validation_end:
].copy()

split_summary = pd.DataFrame(
    {
        "split": [
            "Train",
            "Validation",
            "Test"
        ],
        "rows": [
            len(train_df),
            len(validation_df),
            len(test_df),
        ],
        "start_timestamp": [
            train_df["event_timestamp"].min(),
            validation_df["event_timestamp"].min(),
            test_df["event_timestamp"].min(),
        ],
        "end_timestamp": [
            train_df["event_timestamp"].max(),
            validation_df["event_timestamp"].max(),
            test_df["event_timestamp"].max(),
        ],
        "laundering_rate": [
            train_df["is_laundering"].mean(),
            validation_df["is_laundering"].mean(),
            test_df["is_laundering"].mean(),
        ],
    }
)

split_summary


## 21. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 22. Create persistent project directories

In [ ]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/IBM_AML_Compliance_Project"
)

DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

for directory in [
    DATA_DIR,
    MODEL_DIR,
    REPORT_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directory:", PROJECT_DIR)


## 23. Save prepared datasets

In [ ]:
train_df.to_parquet(
    DATA_DIR / "train.parquet",
    index=False
)

validation_df.to_parquet(
    DATA_DIR / "validation.parquet",
    index=False
)

test_df.to_parquet(
    DATA_DIR / "test.parquet",
    index=False
)

quality_report.to_csv(
    REPORT_DIR / "data_quality_report.csv",
    index=False
)

split_summary.to_csv(
    REPORT_DIR / "split_summary.csv",
    index=False
)

print("Prepared datasets and reports saved.")


## 24. Define baseline model features

In [ ]:
numeric_features = [
    "log_amount_paid",
    "log_amount_difference",
    "is_cross_bank",
    "currency_mismatch",
    "new_counterparty",
    "sender_previous_transaction_count",
    "amount_to_sender_average_ratio",
    "transaction_hour",
    "day_of_week",
    "is_weekend",
]

categorical_features = [
    "payment_format",
    "payment_currency",
    "receiving_currency",
]

target_column = "is_laundering"

all_features = (
    numeric_features
    + categorical_features
)


## 25. Train a Logistic Regression baseline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ]
)

baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                solver="liblinear",
                random_state=42,
            ),
        ),
    ]
)

X_train = train_df[all_features]
y_train = train_df[target_column]

baseline_model.fit(
    X_train,
    y_train
)

print("Baseline model training completed.")


## 26. Evaluate the validation set

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)

X_validation = validation_df[
    all_features
]

y_validation = validation_df[
    target_column
]

validation_probability = (
    baseline_model.predict_proba(
        X_validation
    )[:, 1]
)

DEFAULT_THRESHOLD = 0.50

validation_prediction = (
    validation_probability
    >= DEFAULT_THRESHOLD
).astype(int)

print(
    classification_report(
        y_validation,
        validation_prediction,
        digits=4,
        zero_division=0,
    )
)


In [ ]:
validation_metrics = {
    "threshold": DEFAULT_THRESHOLD,
    "precision": float(
        precision_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        )
    ),
    "recall": float(
        recall_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        )
    ),
    "f1": float(
        f1_score(
            y_validation,
            validation_prediction,
            zero_division=0,
        )
    ),
    "pr_auc": float(
        average_precision_score(
            y_validation,
            validation_probability,
        )
    ),
    "confusion_matrix": (
        confusion_matrix(
            y_validation,
            validation_prediction,
        ).tolist()
    ),
}

validation_metrics


## 27. Plot the precision–recall curve

In [ ]:
precision_values, recall_values, thresholds = (
    precision_recall_curve(
        y_validation,
        validation_probability,
    )
)

plt.figure(figsize=(8, 5))
plt.plot(
    recall_values,
    precision_values
)
plt.title(
    "Validation Precision–Recall Curve"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.tight_layout()
plt.show()


## 28. Select the validation threshold with the best F1 score

In [ ]:
f1_values = (
    2
    * precision_values[:-1]
    * recall_values[:-1]
    / (
        precision_values[:-1]
        + recall_values[:-1]
        + 1e-12
    )
)

best_index = int(
    np.argmax(f1_values)
)

selected_threshold = float(
    thresholds[best_index]
)

print(
    "Selected threshold:",
    round(selected_threshold, 6)
)
print(
    "Validation precision:",
    round(
        float(
            precision_values[best_index]
        ),
        6
    )
)
print(
    "Validation recall:",
    round(
        float(
            recall_values[best_index]
        ),
        6
    )
)
print(
    "Validation F1:",
    round(
        float(
            f1_values[best_index]
        ),
        6
    )
)


## 29. Evaluate the untouched test set

In [ ]:
X_test = test_df[all_features]
y_test = test_df[target_column]

test_probability = (
    baseline_model.predict_proba(
        X_test
    )[:, 1]
)

test_prediction = (
    test_probability
    >= selected_threshold
).astype(int)

test_metrics = {
    "threshold": selected_threshold,
    "precision": float(
        precision_score(
            y_test,
            test_prediction,
            zero_division=0,
        )
    ),
    "recall": float(
        recall_score(
            y_test,
            test_prediction,
            zero_division=0,
        )
    ),
    "f1": float(
        f1_score(
            y_test,
            test_prediction,
            zero_division=0,
        )
    ),
    "pr_auc": float(
        average_precision_score(
            y_test,
            test_probability,
        )
    ),
    "confusion_matrix": (
        confusion_matrix(
            y_test,
            test_prediction,
        ).tolist()
    ),
}

print(
    classification_report(
        y_test,
        test_prediction,
        digits=4,
        zero_division=0,
    )
)

test_metrics


## 30. Build a scored transaction output

In [ ]:
scored_test = test_df[
    [
        "transaction_id",
        "event_timestamp",
        "sender_node_id",
        "receiver_node_id",
        "amount_paid",
        "payment_currency",
        "payment_format",
        "is_laundering",
    ]
].copy()

scored_test[
    "predicted_risk_probability"
] = test_probability

scored_test[
    "predicted_risk_label"
] = test_prediction

scored_test["risk_level"] = pd.cut(
    scored_test[
        "predicted_risk_probability"
    ],
    bins=[
        -0.01,
        0.30,
        0.70,
        1.00
    ],
    labels=[
        "Low",
        "Medium",
        "High"
    ],
)

scored_test.head(20)


## 31. Save the model, metrics and scored transactions

In [ ]:
import json
import joblib

joblib.dump(
    baseline_model,
    MODEL_DIR
    / "baseline_logistic_regression.joblib",
)

with open(
    REPORT_DIR
    / "validation_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        validation_metrics,
        file,
        indent=2,
    )

with open(
    REPORT_DIR
    / "test_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        test_metrics,
        file,
        indent=2,
    )

scored_test.to_parquet(
    DATA_DIR / "scored_test.parquet",
    index=False,
)

print(
    "Model, metrics and scored "
    "transactions saved."
)


# Next stages

After this baseline runs successfully:

1. Add deterministic compliance rules
2. Add 1-hour, 24-hour, 7-day and 30-day behavioural features
3. Add network and graph features
4. Train XGBoost or LightGBM
5. Add Isolation Forest anomaly detection
6. Combine rule, supervised and anomaly scores
7. Build an RBI/PMLA policy corpus
8. Add hybrid retrieval and reranking
9. Use an LLM to explain only validated evidence
10. Add output validation, human review and audit logging

The LLM should not independently calculate or alter the transaction risk score.
